# Clasificador de radiografías de tórax con NIH ChestX-ray14

Red neuronal (DenseNet121 preentrenada, al estilo CheXNet) para detectar **14 patologías torácicas** en una radiografía frontal. Es un problema **multietiqueta**: una misma imagen puede tener varias patologías a la vez (o ninguna), por eso la salida usa `sigmoid` por clase y la pérdida es `BCEWithLogits`.

**Forma de uso**
1. `Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)`.
2. Necesitás una cuenta de Kaggle y un token de API (Kaggle → Settings → API → *Create New Token*). El notebook te pedirá usuario y token.
3. Ejecutá las celdas en orden. Con `USAR_MUESTRA = True` se descarga la muestra del 5% (~5.600 imágenes, ~1 GB) para probar rápido; con `False` se usa el dataset completo (112.120 imágenes, ~45 GB, ocupa mucho disco y tiempo).

> ⚠️ Es un proyecto experimental y de investigación. **No** es un dispositivo médico ni sirve para diagnóstico clínico.

## Instalación e imports

In [ ]:
!pip -q install kagglehub scikit-learn pandas matplotlib

import os, glob, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

def fijar_semilla(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
fijar_semilla()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "sin GPU (será lento)")

## Configuración

In [ ]:
USAR_MUESTRA = True     # True: muestra 5% (rápido). False: dataset completo (~45 GB).
IMG_SIZE     = 224
BATCH_SIZE   = 64
EPOCHS       = 8        # con la muestra podés subir a 15; con el dataset completo 5-10 alcanza
LR           = 1e-4
NUM_WORKERS  = 2
RUTA_MODELO  = "chestxray14_densenet121.pt"

CLASES = ["Atelectasis", "Cardiomegaly", "Effusion", "Infiltration", "Mass", "Nodule",
          "Pneumonia", "Pneumothorax", "Consolidation", "Edema", "Emphysema",
          "Fibrosis", "Pleural_Thickening", "Hernia"]
N_CLASES = len(CLASES)

## Descarga del dataset (Kaggle)

In [ ]:
import kagglehub
kagglehub.login()   # te pide usuario y token de Kaggle

slug = "nih-chest-xrays/sample" if USAR_MUESTRA else "nih-chest-xrays/data"
RUTA_DATOS = kagglehub.dataset_download(slug)
print("Datos en:", RUTA_DATOS)

## Etiquetas e imágenes

In [ ]:
# Archivo de etiquetas (sample_labels.csv en la muestra, Data_Entry_2017.csv en el completo)
csv_path = None
for nombre in ["Data_Entry_2017.csv", "sample_labels.csv"]:
    hits = glob.glob(os.path.join(RUTA_DATOS, "**", nombre), recursive=True)
    if hits:
        csv_path = hits[0]; break
assert csv_path, "No se encontró el CSV de etiquetas"

df = pd.read_csv(csv_path)
df = df.rename(columns={"Image Index": "imagen", "Finding Labels": "hallazgos", "Patient ID": "paciente"})

# Mapa nombre de archivo -> ruta
rutas = {os.path.basename(p): p for p in glob.glob(os.path.join(RUTA_DATOS, "**", "*.png"), recursive=True)}
df["ruta"] = df["imagen"].map(rutas)
df = df.dropna(subset=["ruta"]).reset_index(drop=True)

# Multi-hot: "Effusion|Mass" -> vector de 14 posiciones; "No Finding" -> todo ceros
for c in CLASES:
    df[c] = df["hallazgos"].apply(lambda s, c=c: int(c in s.split("|")))

print(f"{len(df)} imágenes | {df['paciente'].nunique()} pacientes")
prev = df[CLASES].mean().sort_values(ascending=False) * 100
print("\nPrevalencia (%) por clase:\n", prev.round(2).to_string())
print(f"\nSin hallazgos: {(df[CLASES].sum(axis=1) == 0).mean() * 100:.1f}%")

## División train / val / test

Se divide **por paciente** (no por imagen) para evitar fuga de datos: un mismo paciente tiene varias radiografías y si estuvieran repartidas entre train y test, la evaluación saldría inflada.

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
idx_trval, idx_test = next(gss.split(df, groups=df["paciente"]))
df_trval, df_test = df.iloc[idx_trval], df.iloc[idx_test]

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.125, random_state=42)  # 12.5% de 80% = 10% total
idx_tr, idx_val = next(gss2.split(df_trval, groups=df_trval["paciente"]))
df_train, df_val = df_trval.iloc[idx_tr], df_trval.iloc[idx_val]

print(f"train: {len(df_train)} | val: {len(df_val)} | test: {len(df_test)}")

## Dataset y DataLoaders

In [ ]:
MEDIA, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

tf_train = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(8),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(MEDIA, STD),
])
tf_eval = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEDIA, STD),
])

class ChestXrayDataset(Dataset):
    def __init__(self, df, tf):
        self.rutas = df["ruta"].values
        self.y = df[CLASES].values.astype(np.float32)
        self.tf = tf
    def __len__(self): return len(self.rutas)
    def __getitem__(self, i):
        img = Image.open(self.rutas[i]).convert("RGB")   # radiografía en gris -> 3 canales
        return self.tf(img), torch.from_numpy(self.y[i])

dl_train = DataLoader(ChestXrayDataset(df_train, tf_train), batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
dl_val   = DataLoader(ChestXrayDataset(df_val, tf_eval), batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True)
dl_test  = DataLoader(ChestXrayDataset(df_test, tf_eval), batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True)

## Modelo: DenseNet121 con 14 salidas

In [ ]:
def crear_modelo():
    m = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
    m.classifier = nn.Linear(m.classifier.in_features, N_CLASES)   # logits; sigmoid se aplica fuera
    return m

modelo = crear_modelo().to(DEVICE)

# Las clases están muy desbalanceadas: se pondera el término positivo de cada clase
pos = df_train[CLASES].sum().values
neg = len(df_train) - pos
pos_weight = torch.tensor(np.clip(neg / np.maximum(pos, 1), 1, 50), dtype=torch.float32, device=DEVICE)

criterio  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(modelo.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=1)
scaler    = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

## Entrenamiento

In [ ]:
@torch.no_grad()
def predecir_loader(model, loader):
    model.eval()
    probs, ys = [], []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
            logits = model(x)
        probs.append(torch.sigmoid(logits.float()).cpu().numpy()); ys.append(y.numpy())
    return np.concatenate(probs), np.concatenate(ys)

def auc_por_clase(y, p):
    out = {}
    for i, c in enumerate(CLASES):
        out[c] = roc_auc_score(y[:, i], p[:, i]) if 0 < y[:, i].sum() < len(y) else np.nan
    return out

historial = {"loss": [], "auc_val": []}
mejor_auc = 0.0

for epoca in range(1, EPOCHS + 1):
    modelo.train(); t0 = time.time(); perdida_total = 0.0
    for x, y in dl_train:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
            loss = criterio(modelo(x), y)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()
        perdida_total += loss.item()

    p_val, y_val = predecir_loader(modelo, dl_val)
    auc_val = np.nanmean(list(auc_por_clase(y_val, p_val).values()))
    scheduler.step(auc_val)
    historial["loss"].append(perdida_total / len(dl_train)); historial["auc_val"].append(auc_val)

    marca = ""
    if auc_val > mejor_auc:
        mejor_auc = auc_val; torch.save(modelo.state_dict(), RUTA_MODELO); marca = "  ✓ guardado"
    print(f"Época {epoca:02d}/{EPOCHS} | loss {historial['loss'][-1]:.4f} | AUC val medio {auc_val:.4f} | {time.time()-t0:.0f}s{marca}")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(historial["loss"], marker="o"); ax[0].set_title("Pérdida (train)"); ax[0].set_xlabel("época")
ax[1].plot(historial["auc_val"], marker="o"); ax[1].set_title("AUC medio (validación)"); ax[1].set_xlabel("época")
plt.tight_layout(); plt.show()

## Evaluación en test

La métrica estándar en ChestX-ray14 es el **AUC-ROC por patología**. Como referencia, CheXNet (DenseNet121 entrenada sobre el dataset completo) reportó un AUC medio $\approx0.84$. Con la muestra del 5% es normal quedar bastante por debajo ($\approx0.65–0.75$), y clases muy raras como *Hernia* pueden salir muy inestables.

In [ ]:
modelo.load_state_dict(torch.load(RUTA_MODELO, map_location=DEVICE))
p_test, y_test = predecir_loader(modelo, dl_test)

aucs = auc_por_clase(y_test, p_test)
tabla = pd.DataFrame({"AUC": aucs, "positivos_en_test": y_test.sum(0).astype(int)}, index=CLASES).sort_values("AUC", ascending=False)
print(tabla.round(3).to_string())
print(f"\nAUC medio en test: {np.nanmean(list(aucs.values())):.4f}")

tabla["AUC"].plot.barh(figsize=(7, 5), title="AUC por patología (test)").invert_yaxis()
plt.axvline(0.5, color="gray", ls="--"); plt.xlim(0.4, 1.0); plt.tight_layout(); plt.show()

## Inferencia sobre una radiografía + mapa de calor (Grad-CAM)

`clasificar()` devuelve la probabilidad de cada patología. Las probabilidades son independientes entre sí (no suman 1). `umbral` es un criterio simple para marcar una patología como «detectada»; en un uso serio habría que calibrarlo por clase con el conjunto de validación.

In [ ]:
def preprocesar(ruta):
    img = Image.open(ruta).convert("RGB")
    return img, tf_eval(img).unsqueeze(0).to(DEVICE)

@torch.no_grad()
def clasificar(ruta, umbral=0.5, top=5):
    modelo.eval()
    _, x = preprocesar(ruta)
    p = torch.sigmoid(modelo(x)).squeeze().cpu().numpy()
    orden = np.argsort(-p)
    print(f"Imagen: {os.path.basename(ruta)}")
    for i in orden[:top]:
        print(f"  {CLASES[i]:<20} {p[i]*100:5.1f}%  {'◄ detectada' if p[i] >= umbral else ''}")
    if p.max() < umbral:
        print("  -> Ninguna patología supera el umbral (compatible con 'No Finding').")
    return dict(zip(CLASES, p))

def grad_cam(ruta, clase=None):
    modelo.eval()
    img, x = preprocesar(ruta)
    feats = modelo.features(x); feats.retain_grad()
    pooled = F.adaptive_avg_pool2d(F.relu(feats), 1).flatten(1)
    logits = modelo.classifier(pooled)
    idx = int(logits.argmax()) if clase is None else CLASES.index(clase)
    modelo.zero_grad(); logits[0, idx].backward()
    w = feats.grad.mean(dim=(2, 3), keepdim=True)
    cam = F.relu((w * feats).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=img.size[::-1], mode="bilinear", align_corners=False).squeeze().detach().cpu().numpy()
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    prob = torch.sigmoid(logits)[0, idx].item()
    plt.figure(figsize=(5, 5)); plt.imshow(img.convert("L"), cmap="gray"); plt.imshow(cam, cmap="jet", alpha=0.35)
    plt.title(f"{CLASES[idx]}  ({prob*100:.1f}%)"); plt.axis("off"); plt.show()

# Ejemplo: 3 imágenes de test con sus etiquetas reales
for _, fila in df_test.sample(3, random_state=1).iterrows():
    print("Etiqueta real:", fila["hallazgos"])
    clasificar(fila["ruta"]); grad_cam(fila["ruta"]); print("-" * 50)

# Para tu propia imagen (subila a Colab con el panel de archivos):
# clasificar("/content/mi_radiografia.png"); grad_cam("/content/mi_radiografia.png")

## Guardar en Google Drive (opcional)

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# !cp chestxray14_densenet121.pt /content/drive/MyDrive/

## Potenciales mejoras

- Usar el **dataset completo** (`USAR_MUESTRA = False`) y las listas oficiales `train_val_list.txt` / `test_list.txt` para poder comparar con la literatura.
- Subir la resolución a 320–512 px y entrenar más épocas (descongelar todo con LR bajo, o *discriminative learning rates*).
- Probar otras arquitecturas (EfficientNet, ConvNeXt, ViT) o modelos preentrenados en radiografías (TorchXRayVision).
- Calibrar el umbral por clase maximizando F1 o el índice de Youden en validación.
- Tener en cuenta que las etiquetas de ChestX-ray14 fueron extraídas automáticamente de informes con NLP y tienen ruido (se estima ~10 % de error), lo que limita el techo de rendimiento.